## Import Library

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2

## Membuat Petrained Model

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

## Freeze Layer

In [ ]:
base_model.trainable = False

# # Apabila ingin menggunakan fine tunning atau membuat beberapa layer pretained model dapat diubah weightnya
# for layer in base_model.layers[:-20]:
#     layer.trainable = False

## Menambahkan Layer Baru

In [ ]:
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax")
])

## Compile Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## Training

### Load Dataset

In [ ]:
dataset_dir = r"..\..\Datasets\datasets\cats_vs_dogs_extracted\PetImages"

train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

### Cek Label Kelas

In [ ]:
class_names = train_dataset.class_names

print(class_names)

### Normalisasi Gambar

#### Mengubah yang piksel dari 1-255 jadi -1 sampai 1

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_dataset = train_dataset.map(
    lambda x, y: (preprocess_input(x), y)
)

val_dataset = val_dataset.map(
    lambda x, y: (preprocess_input(x), y)
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=2
)

## Evaluasi Model

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title("Training Loss vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("Training Accuracy vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = []
y_pred = []

for images, labels in val_dataset:
    pred = model.predict(images, verbose=0)
    pred = np.argmax(pred, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(pred)

cm = confusion_matrix(y_true, y_pred)

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
).plot(cmap="Blues")

plt.show()

In [ ]:
img_path = r"..\..\Datasets\Uji\kucing.png"

img = tf.keras.utils.load_img(
    img_path,
    target_size=(224, 224)
)

img = tf.keras.utils.img_to_array(img)

img = np.expand_dims(img, axis=0)

img = preprocess_input(img)

pred = model.predict(img, verbose=0)

index = np.argmax(pred)

print("Prediksi :", class_names[index])

for nama, nilai in zip(class_names, pred[0]):
    print(f"{nama}: {nilai*100:.2f}%")